# Vector Databases and Chroma

What kind of datastore?
---
Chroma is a vector database.

Why/when to use?
---
It is useful for similarity search and scaling as the input documents of text/images/etc. are vectorized to allow efficient calculation of distance.

What is Chroma?
---
Chroma is an open source vector database designed to be easy to use locally.

How does Chroma compare to Milvus, Weaviate, and Pinecone?
---
Chroma focuses on ease of use and embeddings while the other three seem to focus on scalability and production deployment. Looking online Milvus has advantages in hybrid search, sharding, multiple index types, data persistence. Weaviate has advantages in building semantic graphs, graphql, modular architecture, and cross-modal search. And Pinecone has advantages in managed service, real-time indexing, and scalability.

# Setup

In [1]:
! pip install chromadb

import chromadb

client = chromadb.Client()

comfymugdb = client.create_collection("answers")

# Data Creation

In [2]:
# Create 8 example documents to work with
documents = [
    "Peru Cusco Santa Teresa is a city roast (light, medium roast) washed process coffee that is fun and complex with pronounced fruit notes (green grape, blackberry) and an interesting cinnamon finish.",
    "Papua New Guinea Baroida Estate (Eastern Highlands) is a city roast (light, medium roast) washed process coffee that is a handpicked microlot sold as a premium offering. It has notes of fresh orange and pineapple flavor with a buttery texture.",
    "Ethiopia Guji a small region that used to be part of Sidamo and produces a dry/natural process coffee with notes of lemondrops, berries, tangerine, chocolate, and almond with a silky mouthfeel.",
    "Rwanda Rukiberuga Farm is a city roast (light, medium roast) natural process coffee with notes of cranberry, toasted sugar dry red wine, and floral undertones.",
    "Sulawesi Toraja Sapan Minanga is a full-city+/vienna roast (dark roast) washed process coffee with notes black pepper, tobacco, and thick syrupy body. Shares similarities to Sumatra.",
    "Ecuador Finca Carolina is a special varietal called Sidra. This washed process coffee is city roast (lightly roasted) and has notes of raspberry, bright florality, sugarcane sweetness, and lime zest finish.",
    "Zambia Kateshi Estate AAA is a city roast+ (medium roast) washed process coffee that has notes of grape, cherry, citrus, and a simply but pleasant brown sugar sweetness.",
    "Guatemala Huehuetenango FECCEG is a full-city roast (medium, dark roast) washed process coffee that has notes of dark chocolate, nuts, and a creamy silky texture.",
]
ids = ["id1", "id2", "id3", "id4", "id5", "id6", "id7", "id8"]

# Create 2 example queries to work with
queries = [
    "I'm looking for a light roast coffee that has bright acidity and floral qualities.",
    "I'm looking for a dark roast coffee that's got notes of bitter chocolate and spices.",
]

# Add the documents to my collection
comfymugdb.add(
    documents=documents,
    ids=ids
)

# Embedding Creation

What does vector embedding means and why is it used?
---
Vector embedding is converting text/images/etc. into a numerical vector and is used to allow computers to run mathematical operations like calucating distance (which captures similarity).

What is an embedding space?
---
It is a theoretical multidimensional space representing all possible embeddings representing the data.

What algorithm did Chroma automatically use in `.add`? Briefly explain what this algorithm is and what it's doing in 1-3 sentences.
---
The algorithm is a sentence transformer model call all-MiniLM-L6-v2. It is essentially a pre-trained transformer model that takes in each document and spits out an embedding vector.

In [3]:
# Print the size of the embedding space
print(comfymugdb.count())

8


# Index Creation

What is indexing in vector databases?
---
Indexing is done so similar queries can be found quickly without have to calculate the query vector against every single document vector in the database.

What does the database do when creating an index?
---
Vectors are partitioned into good structures for searching and some implementations even pre-calculates some of the distances.

What are the main tradeoffs for precomputing vector indexes?
---
The main tradeoffs are storage space, costly updates, and may not be as fresh and accurate.

What algorithm did Chroma automatically use in `.add`? Briefly explain what this algorithm is and what it's doing in 1-3 sentences.
---
Chroma uses an ANN or Approximate kNN algorithm called Hierarchical Navigable Small world and it is apparently created when the collection is queried for the first time and not at the time of `.add`. What it does is traverse a graph like structure where the 'nodes' are vectors and as the distance gets smaller, the more HNSW focuses on finding nearest neighbors.

# Similarity Search

What is similarity search in the context of databases?
---
It is essentially kNN on a massive scale in order to figure out relationships between queries and documents.

What does it mean to embed a query?
---
It means vectorizing it using the same transformer network used to vectorize the documents inorder to get to the same embedding space. Once it becomes a numerical vector you can use l1/l2/cosine/etc. similarity and distance measures to evaluate it.

In [13]:
# Perform a similarity search using `.query` using the 2 queries from step 4
results = comfymugdb.query(
    query_texts=queries,
    n_results=2,
    include=["documents"]
)

# Print the results
print(queries[0],":")
print(results['documents'][0][0])
print(results['documents'][0][1])

print()

print(queries[1],":")
print(results['documents'][1][0])
print(results['documents'][1][1])

I'm looking for a light roast coffee that has bright acidity and floral qualities. :
Zambia Kateshi Estate AAA is a city roast+ (medium roast) washed process coffee that has notes of grape, cherry, citrus, and a simply but pleasant brown sugar sweetness.
Papua New Guinea Baroida Estate (Eastern Highlands) is a city roast (light, medium roast) washed process coffee that is a handpicked microlot sold as a premium offering. It has notes of fresh orange and pineapple flavor with a buttery texture.

I'm looking for a dark roast coffee that's got notes of bitter chocolate and spices. :
Papua New Guinea Baroida Estate (Eastern Highlands) is a city roast (light, medium roast) washed process coffee that is a handpicked microlot sold as a premium offering. It has notes of fresh orange and pineapple flavor with a buttery texture.
Zambia Kateshi Estate AAA is a city roast+ (medium roast) washed process coffee that has notes of grape, cherry, citrus, and a simply but pleasant brown sugar sweetness.

What do these results represent?
---
In the documentation I see they say that the default is l2 norm, or the euclidean distance. Hence you can think of the the results like the first two neighbors in kNN using l2 distance of the embedding space.

How would you use them to respond to a user of the chatbot?
---
Depending on the customer's query, the chatbot can find the most similar coffee name/description and recommend it to them.

# Scale

What are my options for scaling the vector database / chatbot?
---
Vertical and horizontal scaling both seem to be good which probably means migrating to Milvus or Pinecone which seem to have more tools suited for large scale productions/queries.

What considerations/tradeoffs do I need to weigh?
---
Tradeoffs would include the common concerns of network partitions if replications and sharding are taken into consideration. I believe the combination of network parition and sharding also exacerbates the problems of indexing calculations and the freshness of indexes. And since Pinecone was mentioned earlier, I believe offloading it to a managed platform as a SAAS would also be a reasonable option for a smaller business like a coffee shop where an inhouse server room seems less practical.

# [Additional] Repeat Steps using Milvus

Differences between Chroma and Milvus:

In the context of this homework the difference between Chroma dand Milvus isn't that large. In a more involved scenario, Chroma is not as suited for large-scale datasets and has less features than Milvus.

In [5]:
# Install
! pip install pymilvus
! pip install "pymilvus[model]"

In [6]:
# Setup
from pymilvus import MilvusClient, model

client = MilvusClient("comfymug.db")

In [7]:
# Create collection
if client.has_collection("answers"):
    client.drop_collection("answers")

# Keep inline with Chroma's default embedding dimension of 768
client.create_collection("answers", dimension=768)

In [8]:
# Embedding Creation
embedding_fn = model.DefaultEmbeddingFunction()

vectors = embedding_fn.encode_documents(documents)

# Format the data
data = [
    {
        "id": i,
        "vector": vectors[i],
        "text": documents[i]
    }
    for i in range(len(documents))
]

# Insert the data
res = client.insert("answers", data)
print(res)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/827 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

model.onnx:   0%|          | 0.00/46.9M [00:00<?, ?B/s]

{'insert_count': 8, 'ids': [0, 1, 2, 3, 4, 5, 6, 7]}


In [9]:
# Similarity Search
query_vectors = embedding_fn.encode_queries(queries)

results = client.search(
    collection_name="answers",
    data=query_vectors,
    limit=2,
    output_fields=["text"]
)

print(results)

data: ["[{'id': 7, 'distance': 0.5979253649711609, 'entity': {'text': 'Guatemala Huehuetenango FECCEG is a full-city roast (medium, dark roast) washed process coffee that has notes of dark chocolate, nuts, and a creamy silky texture.'}}, {'id': 0, 'distance': 0.5868798494338989, 'entity': {'text': 'Peru Cusco Santa Teresa is a city roast (light, medium roast) washed process coffee that is fun and complex with pronounced fruit notes (green grape, blackberry) and an interesting cinnamon finish.'}}]", "[{'id': 7, 'distance': 0.6467791795730591, 'entity': {'text': 'Guatemala Huehuetenango FECCEG is a full-city roast (medium, dark roast) washed process coffee that has notes of dark chocolate, nuts, and a creamy silky texture.'}}, {'id': 0, 'distance': 0.5519214868545532, 'entity': {'text': 'Peru Cusco Santa Teresa is a city roast (light, medium roast) washed process coffee that is fun and complex with pronounced fruit notes (green grape, blackberry) and an interesting cinnamon finish.'}}]"]